In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [ ]:
loader = PyPDFLoader("../data/ml_fundamentals_rag_sample.pdf")
pdf = loader.load()


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
pdf_chunks = splitter.split_documents(pdf)

In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
vector_stores = Chroma.from_documents(
    documents=pdf_chunks,
    embedding=embedding
)

In [ ]:
query = "building a basic rag pipeline"
data = vector_stores.similarity_search(query=query)

In [ ]:
contexts = ""
for doc in data:
    context+=doc.page_content + "\n"

In [ ]:
llm = ChatGroq(
    groq_api_key=os.getenv("groq_api"), 
    model="openai/gpt-oss-20b"
)

In [ ]:
def get_context(query:str):
    data = vector_stores.similarity_search(query=query)
    context = ""
    for doc in data:
        context+=doc.page_content + "\n"
    return {
        "context": context,
        "query": query
    }

In [ ]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answers based on the context for user question and if you dont know the answer then you can say that "I dont know the answer for the specific question."
    context: {context}
    question: {query}
""")

In [ ]:
rag_chain = get_context | prompt | llm

In [ ]:
res = rag_chain.invoke("give me one common pitfall")

In [ ]:
print(res.content)